# Goal 3 — Armenian Tokenizer Surgery: Grafting onto Qwen2.5-0.5B

**Project:** Armenian Tokenizer Surgery  
**Prerequisites:** Goal 2 complete — trained tokenizers in `./tokenizers_trained/`  
**Target GPU:** A100 40GB (Google Cloud a2-highgpu-1g)

## What this notebook does
1. Loads **Qwen2.5-0.5B** (zero Armenian tokens, fertility 7.83) as the base model
2. Loads the **hy_bpe_32k** Armenian tokenizer (top-ranked from Goal 2, fertility 1.67)
3. Extends Qwen's vocabulary with ~29k new Armenian tokens
4. Initializes the new embeddings with **3 strategies** to compare:
   - **A. Mean-init** — new embeddings = mean of all old embeddings + noise
   - **B. Heuristic-init** — average old embeddings of byte-fallback subtokens (FOCUS method)
   - **C. Nearest-init** — copy the nearest existing token's embedding by cosine similarity
5. Evaluates initial perplexity on Armenian text (before fine-tuning)
6. Compares fertility: original Qwen vs extended tokenizer
7. Saves 3 grafted model directories + `goal3_results.json`

---
**Run cells top to bottom. Use tmux so SSH disconnection doesn't kill training.**

## 0. Environment Setup

In [ ]:
# Uncomment and run once on fresh VM
# !pip install transformers==4.44.0 torch sentencepiece accelerate tabulate bitsandbytes

import os, json, math, time, copy
import torch
import torch.nn.functional as F
import numpy as np
import sentencepiece as spm
from pathlib import Path
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from tabulate import tabulate
    HAS_TAB = True
except ImportError:
    HAS_TAB = False

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Configuration

In [ ]:
import os
from pathlib import Path

# ── Auto-detect repository root ───────────────────────────────────────────
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_MODEL_ID       = "Qwen/Qwen2.5-0.5B"
ARMENIAN_SP_MODEL   = str(REPO_ROOT / "models" / "tokenizers" / "hy_bpe_32k.model")
EVAL_TEXT_PATH      = str(REPO_ROOT / "data" / "sample" / "hy_sample_raw.txt")
OUTPUT_BASE         = str(REPO_ROOT / "outputs" / "goal3_output")

print(f"Repository root: {REPO_ROOT}")
print(f"SP model exists: {os.path.exists(ARMENIAN_SP_MODEL)}")
print(f"Eval text exists: {os.path.exists(EVAL_TEXT_PATH)}")

# ── Compute ────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Eval limits ────────────────────────────────────────────────────────────
PERP_MAX_LINES   = 300
PERP_MAX_LENGTH  = 256
NOISE_SCALE      = 0.01

os.makedirs(OUTPUT_BASE, exist_ok=True)
print(f"Device: {DEVICE}  |  dtype: {DTYPE}")
print(f"Output dir: {OUTPUT_BASE}")


## 2. Load Base Model (Qwen2.5-0.5B)

In [ ]:
print("Loading Qwen2.5-0.5B tokenizer (original, before extension)...")
# Keep an unmodified copy — needed for heuristic / nearest init
old_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
ORIGINAL_VOCAB_SIZE = len(old_tokenizer)
print(f"  Original vocab size: {ORIGINAL_VOCAB_SIZE:,}")

# Working copy — we'll extend this one
print("Loading Qwen2.5-0.5B tokenizer (working copy for extension)...")
ext_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print("\nLoading Qwen2.5-0.5B model weights...")
t0 = time.time()
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
print(f"  Loaded in {time.time()-t0:.1f}s")
print(f"  Parameters: {sum(p.numel() for p in base_model.parameters()):,}")

embed_layer = base_model.get_input_embeddings()
print(f"  Embedding table: {embed_layer.weight.shape}  "
      f"(vocab={embed_layer.weight.shape[0]:,}, hidden={embed_layer.weight.shape[1]})")
print(f"  Tied embeddings: {base_model.config.tie_word_embeddings}")
print(f"  LM head weight is embed weight: "
      f"{base_model.lm_head.weight.data_ptr() == embed_layer.weight.data_ptr()}")

## 3. Load Armenian Tokenizer & Identify New Tokens

In [ ]:
print("Loading Armenian BPE-32k tokenizer...")
arm_sp = spm.SentencePieceProcessor()
arm_sp.load(ARMENIAN_SP_MODEL)
print(f"  Armenian vocab size: {arm_sp.get_piece_size():,}")

# IDs to skip: pad=0, unk=1, bos=2, eos=3, mask=4, and all byte-fallback / special tokens
ARM_SPECIAL_IDS = {arm_sp.pad_id(), arm_sp.unk_id(), arm_sp.bos_id(), arm_sp.eos_id()}
try:
    ARM_SPECIAL_IDS.add(arm_sp.PieceToId("<mask>"))
except:
    pass

qwen_vocab_set = set(ext_tokenizer.get_vocab().keys())

arm_tokens_new      = []   # tokens to add to Qwen vocab
arm_tokens_existing = []   # already present in Qwen
arm_tokens_skipped  = []   # byte tokens / specials — skip

for i in range(arm_sp.get_piece_size()):
    if i in ARM_SPECIAL_IDS:
        continue
    piece = arm_sp.id_to_piece(i)
    # Skip byte-fallback tokens <0xXX> and other angle-bracket specials
    if piece.startswith('<') and piece.endswith('>'):
        arm_tokens_skipped.append(piece)
        continue
    if piece in qwen_vocab_set:
        arm_tokens_existing.append(piece)
    else:
        arm_tokens_new.append(piece)

print(f"\n  New tokens to graft:         {len(arm_tokens_new):,}")
print(f"  Already in Qwen vocab:       {len(arm_tokens_existing):,}")
print(f"  Skipped (byte/special):      {len(arm_tokens_skipped):,}")
print(f"\n  Sample new tokens (first 10): {arm_tokens_new[:10]}")

In [ ]:
# Vocabulary composition breakdown of new tokens
def is_armenian(s):
    s = s.replace('\u2581', '')  # strip SP space marker
    return bool(s) and any(('\u0530' <= c <= '\u058F') or ('\uFB13' <= c <= '\uFB17') for c in s)

arm_count    = sum(1 for t in arm_tokens_new if is_armenian(t))
mixed_count  = sum(1 for t in arm_tokens_new if not is_armenian(t) and t.strip('\u2581'))
space_count  = sum(1 for t in arm_tokens_new if t == '\u2581')

print(f"New token breakdown:")
print(f"  Pure Armenian:  {arm_count:,}  ({arm_count/len(arm_tokens_new)*100:.1f}%)")
print(f"  Mixed/Latin:    {mixed_count:,}  ({mixed_count/len(arm_tokens_new)*100:.1f}%)")
print(f"  Space marker:   {space_count:,}")
print(f"  Total new:      {len(arm_tokens_new):,}")

## 4. Extend Vocabulary & Resize Model

In [ ]:
# Add new Armenian tokens to the working tokenizer
print(f"Adding {len(arm_tokens_new):,} tokens to Qwen tokenizer...")
n_added = ext_tokenizer.add_tokens(arm_tokens_new, special_tokens=False)
NEW_VOCAB_SIZE = len(ext_tokenizer)
print(f"  Actually added:   {n_added:,}")
print(f"  New vocab size:   {NEW_VOCAB_SIZE:,}  (was {ORIGINAL_VOCAB_SIZE:,})")

# Build a lookup: token string → new ID (for init steps)
new_token_id_map = {tok: ext_tokenizer.convert_tokens_to_ids(tok) for tok in arm_tokens_new}
print(f"  ID range for new tokens: {ORIGINAL_VOCAB_SIZE:,} – {NEW_VOCAB_SIZE-1:,}")

# Save the extended tokenizer (shared by all 3 grafted models)
ext_tok_path = os.path.join(OUTPUT_BASE, "extended_tokenizer")
ext_tokenizer.save_pretrained(ext_tok_path)
print(f"  Extended tokenizer saved → {ext_tok_path}")

In [ ]:
# Resize base_model's embedding table to the new vocab size.
# resize_token_embeddings handles tied embeddings automatically:
# it grows both embed_tokens and lm_head.weight together.
print("Resizing base model embedding table...")
base_model.resize_token_embeddings(NEW_VOCAB_SIZE)

new_embed = base_model.get_input_embeddings()
print(f"  New embedding table: {new_embed.weight.shape}")
print(f"  LM head shape:       {base_model.lm_head.weight.shape}")
print(f"  Still tied:          "
      f"{base_model.lm_head.weight.data_ptr() == new_embed.weight.data_ptr()}")

# Cache old embeddings on CPU for use in heuristic / nearest init
# Shape: [ORIGINAL_VOCAB_SIZE, hidden_size]
old_embeddings_cpu = new_embed.weight.data[:ORIGINAL_VOCAB_SIZE].clone().cpu().float()
print(f"  Cached old embeddings: {old_embeddings_cpu.shape}  "
      f"({old_embeddings_cpu.element_size() * old_embeddings_cpu.numel() / 1e6:.1f} MB)")

## 5. Embedding Initialization Strategies

All three strategies operate only on the **new** rows `[ORIGINAL_VOCAB_SIZE : NEW_VOCAB_SIZE]`.
The first `ORIGINAL_VOCAB_SIZE` rows are **frozen** (unchanged from the pretrained model).

### Strategy overview
| Strategy | Idea | Strength | Weakness |
|----------|------|----------|----------|
| **Mean-init** | mean of all old embeddings + Gaussian noise | Simple, stable | No semantic signal; all new tokens start nearly identical |
| **Heuristic-init** (FOCUS) | average embeddings of the byte-fallback subtokens | Preserves surface-form semantics via old tokenizer | Averaging loses distinctions; byte embeddings are noisy for non-Latin |
| **Nearest-init** | find closest existing token by cosine sim; copy it | Hard assignment — no averaging artifacts | Nearest neighbor may be semantically irrelevant |

In [ ]:
def token_surface_for_qwen(piece: str) -> str:
    """
    Convert an SP piece string to a form the OLD Qwen tokenizer can encode.
    SP uses U+2581 (▁) as its space prefix; Qwen uses byte-level BPE.
    We replace ▁ with a real space so Qwen sees a word boundary.
    """
    surface = piece.replace('\u2581', ' ')
    return surface if surface.strip() else piece  # guard against piece == '▁'


def get_heuristic_embedding(piece: str, old_tok, old_embeds: torch.Tensor) -> torch.Tensor:
    """
    Encode the surface form of `piece` with the OLD Qwen tokenizer,
    then return the mean of those subtoken embeddings.
    Falls back to global mean if encoding produces no valid IDs.
    """
    surface = token_surface_for_qwen(piece)
    old_ids = old_tok.encode(surface, add_special_tokens=False)
    # Filter IDs that are within the old embedding range
    valid_ids = [i for i in old_ids if 0 <= i < old_embeds.shape[0]]
    if valid_ids:
        return old_embeds[valid_ids].mean(dim=0)
    return old_embeds.mean(dim=0)  # global mean fallback


def verify_old_rows_unchanged(model, old_embeds_cpu, label=""):
    """Sanity-check: confirm that the first ORIGINAL_VOCAB_SIZE rows were not modified."""
    current = model.get_input_embeddings().weight.data[:ORIGINAL_VOCAB_SIZE].cpu().float()
    max_diff = (current - old_embeds_cpu).abs().max().item()
    status = "OK" if max_diff < 1e-4 else f"WARN (max diff={max_diff:.6f})"
    print(f"  [{label}] Old row integrity check: {status}")


print("Helper functions defined.")

### Strategy A: Mean-init

In [ ]:
print("=" * 60)
print("STRATEGY A: Mean-init")
print("=" * 60)

# Deep-copy base model so each strategy starts from the same weights
model_mean = copy.deepcopy(base_model)
embed_mean = model_mean.get_input_embeddings()

# Compute mean of OLD embeddings
mean_vec = old_embeddings_cpu.mean(dim=0)  # shape: [hidden_size]
print(f"  Mean embedding norm: {mean_vec.norm().item():.4f}")

# Initialize all new rows: mean + small Gaussian noise
n_new = NEW_VOCAB_SIZE - ORIGINAL_VOCAB_SIZE
noise = torch.randn(n_new, mean_vec.shape[0]) * NOISE_SCALE
new_embeds_a = mean_vec.unsqueeze(0).expand(n_new, -1) + noise

with torch.no_grad():
    embed_mean.weight.data[ORIGINAL_VOCAB_SIZE:] = new_embeds_a.to(
        dtype=embed_mean.weight.dtype, device=embed_mean.weight.device
    )

verify_old_rows_unchanged(model_mean, old_embeddings_cpu, label="mean-init")

# Save
mean_save_path = os.path.join(OUTPUT_BASE, "mean_init")
model_mean.save_pretrained(mean_save_path)
ext_tokenizer.save_pretrained(mean_save_path)
print(f"  Saved → {mean_save_path}")

new_norms_a = embed_mean.weight.data[ORIGINAL_VOCAB_SIZE:].float().norm(dim=1)
print(f"  New token embedding norms: mean={new_norms_a.mean():.4f}, "
      f"std={new_norms_a.std():.4f}, min={new_norms_a.min():.4f}")

### Strategy B: Heuristic-init (FOCUS)

For each new Armenian token `t`, encode its surface form with the **old** Qwen tokenizer  
(which will byte-fallback Armenian characters into `\xXX` byte tokens), then average those  
old byte-token embeddings. This gives a semantically-informed starting point that encodes  
the character composition of the token.

Reference: Dobler & de Melo, *FOCUS: Effective Embedding Initialization for Monolingual  
Specialization of Multilingual Models*, EMNLP 2023.

In [ ]:
print("=" * 60)
print("STRATEGY B: Heuristic-init (FOCUS)")
print("=" * 60)

model_heur = copy.deepcopy(base_model)
embed_heur = model_heur.get_input_embeddings()
device_h   = embed_heur.weight.device
dtype_h    = embed_heur.weight.dtype

fallback_count = 0  # how many tokens fell back to global mean

t0 = time.time()
with torch.no_grad():
    for i, piece in enumerate(arm_tokens_new):
        new_id = new_token_id_map[piece]
        heur_vec = get_heuristic_embedding(piece, old_tokenizer, old_embeddings_cpu)

        # Check if we fell back (heuristic == global mean)
        surface = token_surface_for_qwen(piece)
        if not old_tokenizer.encode(surface, add_special_tokens=False):
            fallback_count += 1

        embed_heur.weight.data[new_id] = heur_vec.to(dtype=dtype_h, device=device_h)

        if (i + 1) % 5000 == 0:
            print(f"  {i+1:,} / {len(arm_tokens_new):,} tokens initialized  "
                  f"({time.time()-t0:.1f}s elapsed)")

elapsed = time.time() - t0
print(f"  Completed {len(arm_tokens_new):,} tokens in {elapsed:.1f}s")
print(f"  Fallback to global mean: {fallback_count:,} tokens")

verify_old_rows_unchanged(model_heur, old_embeddings_cpu, label="heuristic-init")

heur_save_path = os.path.join(OUTPUT_BASE, "heuristic_init")
model_heur.save_pretrained(heur_save_path)
ext_tokenizer.save_pretrained(heur_save_path)
print(f"  Saved → {heur_save_path}")

new_norms_b = embed_heur.weight.data[ORIGINAL_VOCAB_SIZE:].float().norm(dim=1)
print(f"  New token embedding norms: mean={new_norms_b.mean():.4f}, "
      f"std={new_norms_b.std():.4f}, min={new_norms_b.min():.4f}")

### Strategy C: Nearest-init

Same heuristic embedding as Strategy B, but instead of **using** that averaged vector directly,  
find the **closest existing token** by cosine similarity and **copy** its embedding.  
This avoids the averaging artifact (all new tokens becoming similar to each other) by  
anchoring each new token to a single well-trained existing token.

In [ ]:
print("=" * 60)
print("STRATEGY C: Nearest-init")
print("=" * 60)

model_near = copy.deepcopy(base_model)
embed_near = model_near.get_input_embeddings()
device_n   = embed_near.weight.device
dtype_n    = embed_near.weight.dtype

# Pre-normalize old embeddings for fast batched cosine similarity
print("  Pre-normalizing old embeddings for cosine similarity...")
old_norm = F.normalize(old_embeddings_cpu, dim=1)  # [orig_vocab, hidden]

# Process in batches to avoid OOM
BATCH_SIZE = 512
nearest_ids_log = []   # for analysis

t0 = time.time()
with torch.no_grad():
    batch_pieces = []
    batch_new_ids = []
    batch_heur_vecs = []

    def flush_batch():
        if not batch_heur_vecs:
            return
        batch_mat = torch.stack(batch_heur_vecs)          # [B, hidden]
        batch_norm = F.normalize(batch_mat, dim=1)         # [B, hidden]
        sims = torch.mm(batch_norm, old_norm.T)            # [B, orig_vocab]
        neighbors = sims.argmax(dim=1)                     # [B]
        for j, (new_id, nbr_id) in enumerate(zip(batch_new_ids, neighbors.tolist())):
            embed_near.weight.data[new_id] = old_embeddings_cpu[nbr_id].to(
                dtype=dtype_n, device=device_n
            )
            nearest_ids_log.append((batch_pieces[j], old_tokenizer.convert_ids_to_tokens(nbr_id)))
        batch_pieces.clear(); batch_new_ids.clear(); batch_heur_vecs.clear()

    for i, piece in enumerate(arm_tokens_new):
        new_id = new_token_id_map[piece]
        heur_vec = get_heuristic_embedding(piece, old_tokenizer, old_embeddings_cpu)
        batch_pieces.append(piece)
        batch_new_ids.append(new_id)
        batch_heur_vecs.append(heur_vec)

        if len(batch_heur_vecs) == BATCH_SIZE:
            flush_batch()

        if (i + 1) % 5000 == 0:
            print(f"  {i+1:,} / {len(arm_tokens_new):,} tokens  ({time.time()-t0:.1f}s)")

    flush_batch()  # last partial batch

elapsed = time.time() - t0
print(f"  Completed in {elapsed:.1f}s")
print(f"  Sample nearest-neighbor mappings (Armenian → Qwen token):")
for arm_tok, qwen_tok in nearest_ids_log[:10]:
    print(f"    '{arm_tok}' → '{qwen_tok}'")

verify_old_rows_unchanged(model_near, old_embeddings_cpu, label="nearest-init")

near_save_path = os.path.join(OUTPUT_BASE, "nearest_init")
model_near.save_pretrained(near_save_path)
ext_tokenizer.save_pretrained(near_save_path)
print(f"  Saved → {near_save_path}")

new_norms_c = embed_near.weight.data[ORIGINAL_VOCAB_SIZE:].float().norm(dim=1)
print(f"  New token embedding norms: mean={new_norms_c.mean():.4f}, "
      f"std={new_norms_c.std():.4f}, min={new_norms_c.min():.4f}")

## 6. Embedding Norm Comparison

A sanity check: how do new-token embedding norms compare to the old-token distribution?  
Extreme differences (much higher or lower) indicate potential training instability in Goal 4.

In [ ]:
old_norms = old_embeddings_cpu.norm(dim=1)

print("Embedding norm statistics:")
print(f"  OLD tokens  [{ORIGINAL_VOCAB_SIZE:,}]:  "
      f"mean={old_norms.mean():.4f}  std={old_norms.std():.4f}  "
      f"min={old_norms.min():.4f}  max={old_norms.max():.4f}")

for label, norms in [("Mean-init (A)", new_norms_a.cpu()),
                     ("Heuristic-init (B)", new_norms_b.cpu()),
                     ("Nearest-init (C)", new_norms_c.cpu())]:
    print(f"  {label} [{len(norms):,}]:  "
          f"mean={norms.mean():.4f}  std={norms.std():.4f}  "
          f"min={norms.min():.4f}  max={norms.max():.4f}")

## 7. Fertility Comparison

Compare how many tokens the **original Qwen tokenizer** vs the **extended tokenizer**  
(using the Armenian SP model for segmentation) produces on sample Armenian text.

In [ ]:
import unicodedata

def is_armenian_char(c):
    cp = ord(c)
    return (0x0530 <= cp <= 0x058F) or (0xFB13 <= cp <= 0xFB17)

def load_eval_lines(path, max_lines=None):
    lines = []
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for i, line in enumerate(f):
            if max_lines and i >= max_lines:
                break
            line = unicodedata.normalize('NFC', line.strip())
            if len(line) < 15:
                continue
            arm_c = sum(1 for c in line if is_armenian_char(c))
            alpha = sum(1 for c in line if c.isalpha())
            if alpha > 0 and arm_c >= 10 and (arm_c / alpha) >= 0.3:
                lines.append(line)
    return lines

print("Loading evaluation lines...")
eval_lines = load_eval_lines(EVAL_TEXT_PATH)
print(f"  Loaded {len(eval_lines):,} lines")
print(f"  Total words: {sum(len(l.split()) for l in eval_lines):,}")

In [ ]:
def fertility_qwen_original(lines):
    """Fertility using the ORIGINAL Qwen tokenizer (byte-level BPE, no Armenian tokens)."""
    total_tokens, total_words = 0, 0
    for line in lines:
        words = [w for w in line.split() if any(c.isalpha() for c in w)]
        total_words += len(words)
        total_tokens += len(old_tokenizer.encode(line, add_special_tokens=False))
    return total_tokens / total_words if total_words else float('inf'), total_tokens, total_words


def fertility_armenian_sp(lines, sp_model):
    """Fertility using the Armenian SP tokenizer (as used for grafted model input)."""
    total_tokens, total_words = 0, 0
    for line in lines:
        words = [w for w in line.split() if any(c.isalpha() for c in w)]
        total_words += len(words)
        total_tokens += len(sp_model.encode(line))
    return total_tokens / total_words if total_words else float('inf'), total_tokens, total_words


# Use first 2000 lines for speed
FERT_LINES = eval_lines[:2000]

print("Computing fertility on 2,000 lines...")

fert_orig, tok_orig, word_orig = fertility_qwen_original(FERT_LINES)
print(f"  Original Qwen tokenizer: {fert_orig:.2f} tokens/word  "
      f"({tok_orig:,} tokens / {word_orig:,} words)")

fert_arm, tok_arm, word_arm = fertility_armenian_sp(FERT_LINES, arm_sp)
print(f"  Armenian SP (bpe_32k):   {fert_arm:.2f} tokens/word  "
      f"({tok_arm:,} tokens / {word_arm:,} words)")

reduction = (fert_orig - fert_arm) / fert_orig * 100
print(f"\n  Token count reduction: {reduction:.1f}%  "
      f"(grafted model processes {fert_orig/fert_arm:.1f}x fewer tokens per word)")

In [ ]:
# Demo: show tokenization of a few example sentences
demo_sentences = [
    "Հայաստանը Հարավային Կովկասի լեռնային երկիր է։",
    "Երևանը Հայաստանի մայրաքաղաքն ու խոշորագույն քաղաքն է:",
    eval_lines[0] if eval_lines else "",
]
demo_sentences = [s for s in demo_sentences if s]

print("Tokenization comparison on sample sentences:")
print("=" * 80)
for sent in demo_sentences[:3]:
    orig_toks = old_tokenizer.encode(sent, add_special_tokens=False)
    arm_pieces = arm_sp.encode(sent, out_type=str)

    print(f"\nText: {sent[:80]}")
    print(f"  Original Qwen ({len(orig_toks)} tokens): "
          f"{old_tokenizer.convert_ids_to_tokens(orig_toks[:20])}")
    print(f"  Armenian SP   ({len(arm_pieces)} tokens): "
          f"{arm_pieces[:20]}")
    print(f"  Ratio: {len(orig_toks)/max(1,len(arm_pieces)):.1f}x more tokens with original Qwen")

## 8. Perplexity Evaluation (Pre-Fine-Tuning)

Compute perplexity on Armenian text **before any fine-tuning**.

We use the Armenian SP tokenizer to segment text, then map each piece to its ID  
in the extended vocabulary. This measures how well each init strategy starts —  
the lower the perplexity, the better the starting point for Goal 4 LoRA fine-tuning.

**Expected results:** All three will be extremely high (thousands) before fine-tuning.  
The relative ordering between strategies is what matters for comparing Goal 4 recovery speed.

In [ ]:
ext_vocab = ext_tokenizer.get_vocab()  # token_str -> ID lookup

def armenian_encode_ids(text: str) -> list:
    """
    Tokenize Armenian text using the SP model, then map each piece to its
    ID in the extended Qwen vocabulary.
    Pieces not in the extended vocab fall back to Qwen's byte-level encoding.
    """
    pieces = arm_sp.encode(text, out_type=str)
    ids = []
    if ext_tokenizer.bos_token_id is not None:
        ids.append(ext_tokenizer.bos_token_id)
    for p in pieces:
        if p in ext_vocab:
            ids.append(ext_vocab[p])
        else:
            # Byte-level fallback via old Qwen tokenizer
            ids.extend(old_tokenizer.encode(p, add_special_tokens=False))
    if ext_tokenizer.eos_token_id is not None:
        ids.append(ext_tokenizer.eos_token_id)
    return ids


def compute_perplexity(model, text_lines, max_lines=200, max_length=256):
    """
    Compute token-level perplexity on Armenian text.
    Returns (perplexity, total_tokens_evaluated).
    """
    model.eval()
    device = next(model.parameters()).device
    total_nll, total_tokens = 0.0, 0

    with torch.no_grad():
        for line in text_lines[:max_lines]:
            ids = armenian_encode_ids(line)
            if len(ids) < 4:
                continue

            # Chunk to max_length to avoid OOM
            for start in range(0, len(ids), max_length):
                chunk = ids[start : start + max_length]
                if len(chunk) < 2:
                    break
                t = torch.tensor([chunk], dtype=torch.long, device=device)
                try:
                    out = model(t, labels=t)
                    loss = out.loss
                    if not (torch.isnan(loss) or torch.isinf(loss)):
                        n = t.shape[1] - 1
                        total_nll += loss.item() * n
                        total_tokens += n
                except RuntimeError:
                    pass

    if total_tokens == 0:
        return float('inf'), 0
    avg_nll = total_nll / total_tokens
    return math.exp(min(avg_nll, 700)), total_tokens  # cap to prevent overflow


print("Perplexity helpers defined.")
print(f"Will evaluate on {PERP_MAX_LINES} lines with max chunk length {PERP_MAX_LENGTH}.")

In [ ]:
perp_results = {}

for label, model in [
    ("A: Mean-init",      model_mean),
    ("B: Heuristic-init", model_heur),
    ("C: Nearest-init",   model_near),
]:
    print(f"\nEvaluating {label}...")
    t0 = time.time()
    ppl, n_toks = compute_perplexity(
        model, eval_lines,
        max_lines=PERP_MAX_LINES,
        max_length=PERP_MAX_LENGTH,
    )
    elapsed = time.time() - t0
    perp_results[label] = {"perplexity": ppl, "tokens_evaluated": n_toks}
    print(f"  Perplexity: {ppl:,.1f}  |  Tokens: {n_toks:,}  |  Time: {elapsed:.1f}s")

print("\n" + "=" * 60)
print("PERPLEXITY COMPARISON (pre-fine-tuning)")
print("=" * 60)
rows = []
for label, r in perp_results.items():
    rows.append([label, f"{r['perplexity']:,.1f}", f"{r['tokens_evaluated']:,}"])

if HAS_TAB:
    print(tabulate(rows, headers=["Strategy", "Perplexity", "Tokens eval'd"], tablefmt="grid"))
else:
    for row in rows:
        print("  ".join(str(v).ljust(30) for v in row))

best_label = min(perp_results, key=lambda k: perp_results[k]["perplexity"])
print(f"\nLowest pre-FT perplexity: {best_label}")

## 9. Results Summary

In [ ]:
print("\n" + "=" * 70)
print("GOAL 3 COMPLETE — SUMMARY")
print("=" * 70)

print(f"""
Base model:       {BASE_MODEL_ID}
Armenian tok:     {ARMENIAN_SP_MODEL}

Vocabulary surgery:
  Original vocab:  {ORIGINAL_VOCAB_SIZE:,}
  New tokens added:{n_added:,}
  Extended vocab:  {NEW_VOCAB_SIZE:,}

Fertility:
  Original Qwen:   {fert_orig:.2f} tokens/word
  Armenian SP:     {fert_arm:.2f} tokens/word
  Reduction:       {reduction:.1f}%  ({fert_orig/fert_arm:.1f}x fewer tokens)
""")

print("Perplexity (pre-fine-tuning):")
for label, r in perp_results.items():
    marker = " ← best" if label == best_label else ""
    print(f"  {label}: {r['perplexity']:,.1f}{marker}")

print(f"""
Saved models:
  {mean_save_path}
  {heur_save_path}
  {near_save_path}
  {ext_tok_path}  (shared tokenizer)
""")

In [ ]:
# Save all results to goal3_results.json
results = {
    "base_model": BASE_MODEL_ID,
    "armenian_sp_model": ARMENIAN_SP_MODEL,
    "vocabulary": {
        "original_vocab_size": ORIGINAL_VOCAB_SIZE,
        "new_tokens_added": n_added,
        "extended_vocab_size": NEW_VOCAB_SIZE,
        "armenian_token_count": arm_count,
        "tokens_already_in_qwen": len(arm_tokens_existing),
        "byte_tokens_skipped": len(arm_tokens_skipped),
    },
    "fertility": {
        "original_qwen": round(fert_orig, 4),
        "armenian_sp_bpe32k": round(fert_arm, 4),
        "reduction_pct": round(reduction, 2),
        "speedup_factor": round(fert_orig / fert_arm, 2),
        "lines_evaluated": len(FERT_LINES),
    },
    "embedding_norms": {
        "old_tokens_mean": round(old_norms.mean().item(), 4),
        "old_tokens_std":  round(old_norms.std().item(), 4),
        "mean_init_mean":  round(new_norms_a.mean().item(), 4),
        "heuristic_init_mean": round(new_norms_b.mean().item(), 4),
        "nearest_init_mean": round(new_norms_c.mean().item(), 4),
    },
    "perplexity_pre_finetuning": {
        k: {"perplexity": round(v["perplexity"], 1), "tokens_evaluated": v["tokens_evaluated"]}
        for k, v in perp_results.items()
    },
    "best_init_strategy_by_perplexity": best_label,
    "saved_models": {
        "mean_init":      mean_save_path,
        "heuristic_init": heur_save_path,
        "nearest_init":   near_save_path,
        "extended_tokenizer": ext_tok_path,
    },
    "nearest_init_sample_mappings": [
        {"armenian_token": a, "nearest_qwen_token": q}
        for a, q in nearest_ids_log[:50]
    ],
}

results_path = os.path.join(OUTPUT_BASE, "goal3_results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Results saved → {results_path}")
print("\n--- Goal 3 complete! Ready for Goal 4 (LoRA fine-tuning). ---")

---
## Appendix: Nearest-Neighbor Mapping Analysis

Inspect which existing Qwen tokens the nearest-init strategy mapped Armenian tokens to.  
This reveals whether the cosine-similarity search found linguistically plausible neighbors.

In [ ]:
# Show a random sample of 30 Armenian → nearest Qwen token mappings
import random
random.seed(SEED)
sample = random.sample(nearest_ids_log, min(30, len(nearest_ids_log)))

print("Random sample of Nearest-init mappings:")
print(f"{'Armenian token':<25} {'Nearest Qwen token':<25}")
print("-" * 50)
for arm_t, qwen_t in sorted(sample, key=lambda x: x[0]):
    print(f"  {repr(arm_t):<23} → {repr(qwen_t):<23}")

# Check uniqueness: how many distinct Qwen tokens are used as nearest neighbors?
qwen_neighbor_tokens = [q for _, q in nearest_ids_log]
unique_neighbors = len(set(qwen_neighbor_tokens))
print(f"\nOut of {len(nearest_ids_log):,} new Armenian tokens:")
print(f"  Unique Qwen tokens used as neighbors: {unique_neighbors:,}")
print(f"  Coverage ratio: {unique_neighbors/len(nearest_ids_log)*100:.1f}%")
print(f"  (100% = every Armenian token maps to a different Qwen token — ideal diversity)")

In [ ]:
# Check saved model sizes on disk
print("Saved model directory sizes:")
for path in [mean_save_path, heur_save_path, near_save_path, ext_tok_path]:
    total = sum(os.path.getsize(os.path.join(r, f))
                for r, _, files in os.walk(path) for f in files)
    print(f"  {path:<45} {total/1e9:.2f} GB")

In [ ]:
import shutil
shutil.make_archive('/content/goal3_output', 'zip', '/content/goal3_output')